# 03 | Applications

This notebook focuses on the main project questions:

- Which profiles concentrate high ratings?
- Which profiles concentrate low ratings?
- How does the concentration story connect back to the popularity/prestige features from the first study?


## Setup

This notebook depends on two analysis stages:

- the **rating-extremes stage**
- the **feature-alignment stage**

If the required artifacts are missing, it rebuilds them in sequence.


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "rec_dating_project").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


project_root = locate_project_root()
sys.path.insert(0, str(project_root / "src"))

from rec_dating_project import ProjectPaths


paths = ProjectPaths.default()
paths.ensure_output_dirs()
OUTPUT_DATA = paths.output_data_dir
OUTPUT_FIGURES = paths.output_figures_dir
FORCE_REBUILD = False


def run_script(script_name: str, *args: str) -> None:
    cmd = [sys.executable, str(paths.scripts_dir / script_name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=paths.project_root)


def ensure_stage_two_outputs(force: bool = False) -> None:
    required = [
        OUTPUT_DATA / "profile_metrics_full.csv",
        OUTPUT_DATA / "profile_metrics_positive_8_full.csv",
        OUTPUT_DATA / "rater_metrics_full.csv",
    ]
    if force or any(not path.exists() for path in required):
        run_script("02_full_project_analysis.py")


def ensure_stage_three_outputs(force: bool = False) -> None:
    ensure_stage_two_outputs(force=force)

    extremes_required = [
        OUTPUT_DATA / "profile_rating_extremes_summary_full.csv",
        OUTPUT_DATA / "profile_rating_extremes_buckets_full.csv",
        OUTPUT_DATA / "profile_rating_extremes_profiles_full.csv",
        OUTPUT_DATA / "top_profiles_high_ratings_full.csv",
        OUTPUT_DATA / "top_profiles_low_ratings_full.csv",
        OUTPUT_FIGURES / "profile_high_low_rating_concentration_full.png",
        OUTPUT_FIGURES / "profile_rating_concentration_curves_full.png",
        OUTPUT_FIGURES / "profile_bucket_shares_full.png",
        OUTPUT_FIGURES / "profile_interaction_concentration_curve_full.png",
        OUTPUT_FIGURES / "profile_interaction_bucket_shares_full.png",
    ]
    if force or any(not path.exists() for path in extremes_required):
        run_script("03_profile_rating_extremes.py")

    alignment_required = [
        OUTPUT_DATA / "profile_feature_alignment_summary_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_overlap_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_correlations_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_profiles_full.csv",
        OUTPUT_DATA / "profile_feature_alignment_feature_classes_full.csv",
        OUTPUT_FIGURES / "profile_feature_alignment_heatmap_full.png",
        OUTPUT_FIGURES / "profile_feature_alignment_zscore_heatmap_full.png",
        OUTPUT_FIGURES / "profile_feature_alignment_consistency_full.png",
    ]
    if force or any(not path.exists() for path in alignment_required):
        run_script("04_profile_feature_alignment.py")


ensure_stage_three_outputs(FORCE_REBUILD)

extreme_summary = pd.read_csv(OUTPUT_DATA / "profile_rating_extremes_summary_full.csv")
extreme_buckets = pd.read_csv(OUTPUT_DATA / "profile_rating_extremes_buckets_full.csv")
top_high = pd.read_csv(OUTPUT_DATA / "top_profiles_high_ratings_full.csv")
top_low = pd.read_csv(OUTPUT_DATA / "top_profiles_low_ratings_full.csv")

alignment_summary = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_summary_full.csv")
alignment_overlap = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_overlap_full.csv")
alignment_correlations = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_correlations_full.csv")
alignment_profiles = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_profiles_full.csv")
alignment_feature_classes = pd.read_csv(OUTPUT_DATA / "profile_feature_alignment_feature_classes_full.csv")

all_summary = extreme_summary.loc[extreme_summary["series"] == "all_interactions"].iloc[0]
high_summary = extreme_summary.loc[extreme_summary["series"] == "high_ratings"].iloc[0]
low_summary = extreme_summary.loc[extreme_summary["series"] == "low_ratings"].iloc[0]

bucket_order = ["Top 1%", "Top 1-5%", "Top 5-10%", "Top 10-20%", "Top 20-50%", "Bottom 50%"]
rating_bucket_view = (
    extreme_buckets.pivot(index="bucket", columns="series", values="value_share")
    .reindex(bucket_order)
    .rename(columns={"high_ratings": "high_ratings_share", "low_ratings": "low_ratings_share"})
    [["high_ratings_share", "low_ratings_share"]]
    .reset_index()
)
interaction_bucket_view = (
    extreme_buckets.loc[extreme_buckets["series"] == "all_interactions", ["bucket", "value_share"]]
    .rename(columns={"value_share": "all_interactions_share"})
    .set_index("bucket")
    .reindex(bucket_order)
    .reset_index()
)


## Application 1 | Rating Extremes and Interaction Concentration

We start with the profile side:

- overall interaction concentration
- high-rating concentration
- low-rating concentration

This tells us whether the same small subset of profiles dominates multiple kinds of received attention.


In [ ]:
concentration_summary = pd.DataFrame(
    [
        {"series": "all_interactions", "top_1pct_share": all_summary["top_1pct_share"], "top_10pct_share": all_summary["top_10pct_share"], "top_20pct_share": all_summary["top_20pct_share"], "user_share_for_80pct": all_summary["user_share_for_80pct"]},
        {"series": "high_ratings", "top_1pct_share": high_summary["top_1pct_share"], "top_10pct_share": high_summary["top_10pct_share"], "top_20pct_share": high_summary["top_20pct_share"], "user_share_for_80pct": high_summary["user_share_for_80pct"]},
        {"series": "low_ratings", "top_1pct_share": low_summary["top_1pct_share"], "top_10pct_share": low_summary["top_10pct_share"], "top_20pct_share": low_summary["top_20pct_share"], "user_share_for_80pct": low_summary["user_share_for_80pct"]},
    ]
)

display(concentration_summary)
display(rating_bucket_view)
display(interaction_bucket_view)


In [ ]:
concentration_figures = [
    (
        "profile_high_low_rating_concentration_full.png",
        "Cumulative concentration of high and low ratings.",
    ),
    (
        "profile_rating_concentration_curves_full.png",
        "Separate concentration curves for high and low ratings.",
    ),
    (
        "profile_bucket_shares_full.png",
        "Disjoint bucket shares for high and low ratings.",
    ),
    (
        "profile_interaction_concentration_curve_full.png",
        "Overall interaction concentration on the profile side.",
    ),
    (
        "profile_interaction_bucket_shares_full.png",
        "Disjoint bucket shares for overall interaction.",
    ),
]

for filename, note in concentration_figures:
    display(Markdown(f"### {filename}\n\n{note}"))
    display(Image(filename=str(OUTPUT_FIGURES / filename)))


In [ ]:
display(Markdown("### Profiles receiving the most high ratings"))
display(top_high.head(15))

display(Markdown("### Profiles receiving the most low ratings"))
display(top_low.head(15))

display(
    Markdown(
        f"""
        ## Application 1 Takeaways

        - The top **1%** of profiles receives **{all_summary['top_1pct_share']:.2%}** of all interactions.
        - The same top **1%** receives **{high_summary['top_1pct_share']:.2%}** of all high ratings.
        - Low ratings are concentrated too, but less sharply than high ratings.
        - This means that concentration is a general property of the profile side, not only a property of positive ratings.
        """
    )
)


## Application 2 | Feature Alignment Across the Full Bucket Ladder

The next question is whether the concentration study and the popularity/prestige study tell a connected story.

We compare the exact profile buckets from the concentration analysis with the node-level features from the first study.


In [ ]:
overlap_view = alignment_overlap.copy()
overlap_view["bucket"] = pd.Categorical(overlap_view["bucket"], categories=bucket_order, ordered=True)
overlap_view = overlap_view.sort_values("bucket")
display(overlap_view.round(4))

top_overlap = alignment_overlap.loc[alignment_overlap["bucket"] == "Top 1%"].iloc[0]
bottom_overlap = alignment_overlap.loc[alignment_overlap["bucket"] == "Bottom 50%"].iloc[0]

display(
    Markdown(
        f"""
        ### Exact-bucket overlap summary

        - Top 1% overlap: **{top_overlap['intersection_size']:,} shared profiles**, Jaccard **{top_overlap['jaccard_overlap']:.2%}**
        - Bottom 50% overlap: **{bottom_overlap['intersection_size']:,} shared profiles**, Jaccard **{bottom_overlap['jaccard_overlap']:.2%}**
        - Random-null expectation for the Top 1% overlap: **{top_overlap['expected_random_intersection']:.1f}** profiles
        """
    )
)


In [ ]:
alignment_figures = [
    (
        "profile_feature_alignment_heatmap_full.png",
        "Point-biserial correlations between Study 1 features and exact bucket membership.",
    ),
    (
        "profile_feature_alignment_zscore_heatmap_full.png",
        "Standardized feature lift inside each exact bucket.",
    ),
    (
        "profile_feature_alignment_consistency_full.png",
        "Class-level alignment across ALL, POS8, and POS3 layers.",
    ),
]

for filename, note in alignment_figures:
    display(Markdown(f"### {filename}\n\n{note}"))
    display(Image(filename=str(OUTPUT_FIGURES / filename)))


In [ ]:
class_view = alignment_feature_classes[
    [
        "class_label",
        "layer_label",
        "feature_count",
        "interaction_mean_corr",
        "high_mean_corr",
        "included_features",
    ]
].sort_values(["class_label", "layer_label"])

positive_signals = alignment_summary[
    (alignment_summary["interaction_mean_corr"] > 0) & (alignment_summary["high_mean_corr"] > 0)
].sort_values("shared_signal_strength", ascending=False)

display(Markdown("### Class summary"))
display(class_view.round(3))

display(Markdown("### Strongest shared positive signals"))
display(
    positive_signals[
        [
            "feature_label",
            "interaction_mean_corr",
            "high_mean_corr",
            "shared_signal_strength",
        ]
    ].head(8).round(3)
)


In [ ]:
display(
    Markdown(
        f"""
        ## Application 2 Takeaways

        - The alignment is strongest at the extremes of the ranking, especially in the Top 1% and Bottom 50% buckets.
        - The observed Top 1% overlap (**{top_overlap['intersection_size']:,}** profiles) is far larger than the fixed-size random expectation (**{top_overlap['expected_random_intersection']:.1f}**).
        - The most stable shared signals come from degree, strength, authority, and rank-based variables.
        """
    )
)


## Next

The final notebook gathers the last figures and reference values:

- collect the final figures
- show the reference values
- keep the final outputs in one place
